In [ ]:
import os
import pathlib

here = pathlib.Path.cwd()

ROOT = here.parents[2] if here.name == "day03" else here

os.chdir(ROOT)

SANDBOX = ROOT / "sandbox" / "w3" / "day03"

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : c:\workspace\hanwha-agent


In [ ]:
import sys

sys.path.insert(0, "backend")

from app.core.config import get_settings, mask

settings = get_settings()

print("app_mode         :", settings.app_mode)
print("llm_model        :", settings.llm_model)
print("daily_call_limit :", settings.daily_call_limit)
print("max_input_chars  :", settings.max_input_chars)

api_key = settings.anthropic_api_key
print("anthropic_api_key:", mask(api_key.get_secret_value() if api_key else None))

app_mode         : mock
llm_model        : claude-haiku-4-5
daily_call_limit : 200
max_input_chars  : 2000
anthropic_api_key: sk-ant-a...(108자)


In [ ]:
import itertools

class FlakyLLM:

    def __init__(self, answers: list[str]) -> None:
        self._answers = list(answers)
        self._cycle = itertools.cycle(self._answers)
        self.calls = 0

    def ask(self, question: str) -> str:
        self.calls += 1
        return next(self._cycle)

In [ ]:
ANSWERS = [
    "숙박비는 1박 7만원입니다.",
    "국내출장 숙박비 한도는 1박당 7만원입니다.",
    "1박 기준 숙박비는 7만원까지 인정됩니다.",
]
QUESTION = "국내출장 숙박비 한도가 얼마인가요?"
EXPECTED = "숙박비는 1박 7만원입니다."

llm = FlakyLLM(ANSWERS)

for i in range(1, 4):
    answer = llm.ask(QUESTION)
    try:
        assert answer == EXPECTED
        print(f"{i}회차: 통과              | {answer}")
    except AssertionError:
        print(f"{i}회차: 실패 — 기대와 다름 | {answer}")

print()
print(f"기대한 문장 : {EXPECTED}")
print(f"호출 횟수   : {llm.calls}회 — 질문은 한 글자도 바뀌지 않았다")

1회차: 통과              | 숙박비는 1박 7만원입니다.
2회차: 실패 — 기대와 다름 | 국내출장 숙박비 한도는 1박당 7만원입니다.
3회차: 실패 — 기대와 다름 | 1박 기준 숙박비는 7만원까지 인정됩니다.

기대한 문장 : 숙박비는 1박 7만원입니다.
호출 횟수   : 3회 — 질문은 한 글자도 바뀌지 않았다


In [ ]:
DOC_ID = "DOC-HR-014"                       # 문서 번호
FIXED_ANSWER = "숙박비는 1박 7만원입니다"      # 고정된 값
QUESTION = "숙박비 한도가 얼마인가요?"         # 질문


def answer_service(question: str, mode: str) -> str:
    if mode == "mock":                                     
        return f"[{mode}] {FIXED_ANSWER}"
    return f"[{mode}] {FIXED_ANSWER} (진짜 호출을 했다고 치자)"


def draft_service(question: str, mode: str) -> str:
    if mode == "mock":                                      
        return f"[{mode}] {DOC_ID} 근거 신청 초안 (고정)"
    return f"[{mode}] {DOC_ID} 근거 신청 초안 (진짜 호출을 했다고 치자)"


def search_service(question: str, mode: str) -> str:
    if mode == "mock":                                       
        return f"[{mode}] {DOC_ID} 국내출장 여비 규정 v2.0"
    return f"[{mode}] {DOC_ID} 국내출장 여비 규정 v2.0 (진짜 호출을 했다고 치자)"


services = [answer_service, draft_service, search_service]
modes = ["mock", "live"]


for service in services:
    for mode in modes:
        print(f"{service.__name__:<15} {mode:<5} → {service(QUESTION, mode)}")

answer_service  mock  → [mock] 숙박비는 1박 7만원입니다
answer_service  live  → [live] 숙박비는 1박 7만원입니다 (진짜 호출을 했다고 치자)
draft_service   mock  → [mock] DOC-HR-014 근거 신청 초안 (고정)
draft_service   live  → [live] DOC-HR-014 근거 신청 초안 (진짜 호출을 했다고 치자)
search_service  mock  → [mock] DOC-HR-014 국내출장 여비 규정 v2.0
search_service  live  → [live] DOC-HR-014 국내출장 여비 규정 v2.0 (진짜 호출을 했다고 치자)


In [ ]:
# 포트 정의
# - 무엇을 : 결과 타입, 포트, 어댑터 정의
# - 왜 : 뭘 주고 받을지 먼저 선언해두어야 서비스가 누가 해주는지 모를 수 있다.

# dataclasses
# - 변수를 만들면 __init__, __repr__ 결과를 자동으로 만들어줌
from dataclasses import dataclass               
from typing import Protocol, runtime_checkable  


@dataclass
class LLMResult:

    text: str
    model: str
    input_tok: int
    output_tok: int
    cost_krw: float
    latency_ms: int

# 어댑터가 지켜야할 규칙
# - 인터페이스
@runtime_checkable
class LLMPort(Protocol):

    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult: ...

# 고정된 답변만 돌려주는 어댑터
# - 가짜 대답 : LLM을 사용하지 않은 대답
class FixedLLM:                     
    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult:
        return LLMResult(text=FIXED_ANSWER, model="fixed", input_tok=0,
                         output_tok=0, cost_krw=0.0, latency_ms=0)
# 같은 입력에는 같은 결과만 나오도록 테스트 데이터 세팅
USER = {"emp_no": "2019-0412", "name": "김민준", "dept": "인프라사업부 2팀"}
CONTEXTS = [{"doc_id": DOC_ID, "title": "국내출장 여비 규정", "version": "v2.0",
             "security_level": "일반", "file_format": "docx"}]
# 결과 확인
print("FixedLLM 이 LLMPort 를 상속했는가 :", LLMPort in FixedLLM.__mro__)
print("isinstance(FixedLLM(), LLMPort) →", isinstance(FixedLLM(), LLMPort))
print(FixedLLM().answer(question=QUESTION, contexts=CONTEXTS, user=USER))

FixedLLM 이 LLMPort 를 상속했는가 : False
isinstance(FixedLLM(), LLMPort) → True
LLMResult(text='숙박비는 1박 7만원입니다', model='fixed', input_tok=0, output_tok=0, cost_krw=0.0, latency_ms=0)


In [ ]:
# 토큰 어림 계산
# - 과금 단위는 요청 수가 아니라 토큰 수다
# - 한글은 글자당 1토큰 이상, 영문/공백/문장부호는 그보다 적게 잡고 어림한다
# - 진짜 토큰 수는 응답의 usage 로만 알 수 있다 (여기 값은 감을 잡기 위한 추정치)
import math      

def estimate_tokens(text: str) -> int:

    korean = 0    
    other = 0      
    for ch in text:
        
        if "가" <= ch <= "힣":
            korean += 1
        elif ch.isascii():
            other += 1
        else:
            korean += 1

    return math.ceil(korean * 1.0 + other * 0.25)


QUESTION_KO = "국내출장 숙박비 한도가 얼마인가요?"
QUESTION_EN = "What is the lodging limit for a domestic trip?"

for label, text in [("한국어", QUESTION_KO), ("영어  ", QUESTION_EN)]:
    tokens = estimate_tokens(text)
    print(f"{label} · 글자 {len(text):>3}자 · 어림 토큰 {tokens:>3} · 글자당 {tokens / len(text):.2f} 토큰")

한국어 · 글자  19자 · 어림 토큰  16 · 글자당 0.84 토큰
영어   · 글자  46자 · 어림 토큰  12 · 글자당 0.26 토큰


In [ ]:
# 비용 계산
# - 가격은 100만 토큰당 달러로 매겨진다
# - 출력이 입력보다 5배 비싸다 -> 답을 길게 쓰게 만드는 프롬프트가 비용에 더 크게 작용한다
# - cache_write 는 값만 정의해두고 아래 계산식에는 아직 넣지 않았다
PRICE = {                    
    "input": 1.0,
    "cache_write": 1.25,
    "cache_read": 0.10,
    "output": 5.0,
}
USD_KRW = 1400              
IN_TOK, OUT_TOK = 800, 400  
DAILY_CALL_LIMIT = 200      

# 질문 1회 비용 (달러 계산)
def cost_usd(input_tok: int, output_tok: int, cache_read_tok: int = 0) -> float:
    billed_input = input_tok - cache_read_tok
    total = (
        billed_input * PRICE["input"]
        + cache_read_tok * PRICE["cache_read"]
        + output_tok * PRICE["output"]
    ) / 1_000_000           # 숫자 자릿수 구분 _로 연결
    return total

# 원화 계산
def cost_krw(input_tok: int, output_tok: int, cache_read_tok: int = 0) -> float:
    return round(cost_usd(input_tok, output_tok, cache_read_tok) * USD_KRW, 2)

In [ ]:
# 하루치 비용과 캐시 효과 확인
# - DAILY_CALL_LIMIT 는 우리가 먼저 거는 상한이다 (공급자 한도에 부딪히기 전에 멈추려고)
one_call = cost_krw(IN_TOK, OUT_TOK)
print("1회 질의 : ", one_call, "원")

print(f"하루 요청 횟수를 모두 사용하면 : {round(one_call * DAILY_CALL_LIMIT, 2)}원")

cached_call = cost_krw(IN_TOK, OUT_TOK, cache_read_tok=600)
saved = (1 - cached_call / one_call) * 100
print("600이 캐시 읽기 : ", cached_call, "원")
print("절감 : ", saved, "%")

1회 질의 :  3.92 원
하루 요청 횟수를 모두 사용하면 : 784.0원
600이 캐시 읽기 :  3.16 원
절감 :  19.387755102040817 %
